In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/nnott3/KilterTransformer.git
%cd KilterTransformer
!ls

Cloning into 'KilterTransformer'...
remote: Enumerating objects: 282, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 282 (delta 61), reused 51 (delta 24), pack-reused 173 (from 1)
Receiving objects: 100% (282/282), 118.53 MiB | 13.96 MiB/s, done.
Resolving deltas: 100% (134/134), done.
Updating files: 100% (59/59), done.
/content/KilterTransformer
bert_improved.ipynb  gitignore		   main.ipynb	      req
bert.ipynb	     gpt.ipynb		   models	      saved_models
data		     gpt_shuffle.ipynb	   project_structure  src
EDA.ipynb	     gpt_shuffle_me.ipynb  pyproject.toml     utils_old
figs		     gpt_wandb.ipynb	   readme.md	      uv.lock


# init

In [3]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from datetime import datetime
import random
from functools import partial
from typing import List, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    GPT2LMHeadModel,
    GPT2Config,
    PreTrainedTokenizerFast,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
import numpy as np
from datasets import Dataset, disable_progress_bar
import wandb
from transformers import DataCollatorForLanguageModeling
from src.data_processing import DataPreprocessing
from src.tokenizer import train_tokenizer
from src.gpt import preprocess_datasets, KilterGPT
import warnings

warnings.filterwarnings("ignore",)
disable_progress_bar()



In [4]:
wandb.init(
    project="climb-gpt-shuffle",
    name=f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    config={
        "architecture": "GPT-SetLoss",
        "n_embd": 256,
        "n_head": 4,
        "n_layer": 6,
        "n_positions": 128,
        "dropout": 0.1,
        "epochs": 30,
        "batch_size": 16,
        "learning_rate": 1e-4,
        "weight_decay": 0.01,
        "gradient_accumulation_steps": 1,
        "early_stopping_patience": 5,
        "allow_empty_prompt": True,
        "min_prefix_len": 1,  # Changed from 3 to allow BOS-only
    }
)

run_name = wandb.run.name

# OUT_DIR = f"/content/drive/MyDrive/KilterTransformer/models/climb_gpt/{run_name}"
OUT_DIR = f"models/climb_gpt/{run_name}"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: treepatchantaurai (treepatchantaurai-me) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using device: cuda


# data

In [5]:
# Load and split data
dp = DataPreprocessing()
datasets = dp.load_climbs()

# 80-10-10 split
train_test = datasets.train_test_split(test_size=0.2, seed=42)
val_test = train_test['test'].train_test_split(test_size=0.5, seed=42)

datasets = {
    'train': train_test['train'],
    'val': val_test['train'],
    'test': val_test['test']
}

print(f"Train size: {len(datasets['train'])}")
print(f"Val size: {len(datasets['val'])}")
print(f"Test size: {len(datasets['test'])}")

Loaded 76992 routes from cache data/climbs_cleaned.csv
Train size: 61593
Val size: 7699
Test size: 7700


In [6]:
# Train tokenizer
tokenizer = train_tokenizer(datasets, OUT_DIR)
wandb.config.update({"vocab_size": tokenizer.vocab_size})

print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Special tokens: {tokenizer.special_tokens_map}")

# Tokenize datasets
datasets = preprocess_datasets(datasets, tokenizer) # Dataset object
print("✓ Datasets tokenized")

Built vocabulary with 1932 tokens (1928 holds)

Vocab size: 1932 tokens
First 10 tokens: [('feet1229', 652), ('feet1471', 1416), ('start1190', 497), ('hand1579', 1850), ('finish1169', 415), ('hand1245', 718), ('feet1276', 840), ('start1331', 1061), ('angle40', 8), ('feet1390', 1296)]

Sample encodings:

Input: angle35_grade14_feet1595_start1400
Tokens: ['[BOS]', 'angle35', 'grade14', 'feet1595', '[UNK]', '[EOS]', ('[PAD]', 19)]

Input: angle40_grade15_feet1595_start1596_hand1597_finish1598
Tokens: ['[BOS]', 'angle40', 'grade15', 'feet1595', 'start1596', 'hand1597', 'finish1598', '[EOS]', ('[PAD]', 17)]
Saving tokenizer to models/climb_gpt/run_20251104_143357
Vocabulary size: 1932
Special tokens: {'bos_token': '[BOS]', 'eos_token': '[EOS]', 'unk_token': '[UNK]', 'pad_token': '[PAD]'}
✓ Datasets tokenized


In [7]:
example = datasets["train"][0]
for k, v in example.items():
    print(f"{k:15s} -> {v[:10]} ... len={len(v)}")


input_ids       -> [1, 9, 23, 56, 64, 333, 674, 862, 1070, 1122] ... len=13
token_type_ids  -> [0, 0, 0, 0, 0, 0, 0, 0, 0, 0] ... len=13
attention_mask  -> [1, 1, 1, 1, 1, 1, 1, 1, 1, 1] ... len=13


# order-invariant loss
Instead of predicting the *exact next token* (as in normal language modeling),
we want the model to predict **any valid next hold** — since order among holds doesn't matter.

This means at position *i*, the valid next tokens are **all remaining holds** in the sequence.

Shape explanation:
- logits: `(B, L, V)`  → batch_size × seq_len × vocab_size
- labels: `(B, L)`     → each position contains the target token index or -100 for padding

We’ll compute the log-probability of the **set of valid tokens** instead of a single token.



In [ ]:
import torch
import torch.nn.functional as F

# def any_of_next_token_loss(logits, shift_labels, ignore_index=-100, print_debug=False):
#     """
#     logits:   Tensor shape (B, L, V)  -- already shifted logits for predicting next token
#               (e.g. logits[..., :-1, :] from model outputs)
#     shift_labels: Tensor shape (B, L) -- labels shifted left (labels[...,1:])
#                   entries are vocab indices or ignore_index
#     Returns:
#       loss: scalar tensor (mean over valid positions)
#     """
#     B, L, V = logits.shape
#     device = logits.device

#     # Convert logits -> log_probs
#     log_probs = F.log_softmax(logits, dim=-1)  # (B, L, V)

#     # Build valid-mask per position that marks every token that occurs later in the sequence.
#     # For each batch b and position i we want all labels in shift_labels[b, i:] (excluding ignore_index).
#     # We'll build valid_mask by iterating over positions (L is small, so it's fine).
#     valid_mask = torch.zeros((B, L, V), dtype=torch.bool, device=device)

#     for pos in range(L):
#         # tail_labels: shape (B, tail_len)
#         tail_labels = shift_labels[:, pos:]  # (B, L-pos)

#         # flatten to indices with mask of non-ignore
#         flat = tail_labels.reshape(-1)  # (B*(L-pos),)
#         keep = flat != ignore_index
#         if keep.any():
#             kept_indices = flat[keep].long()  # the vocabulary indices we want to mark
#             # To place them back into (B, pos) positions we need which batch each came from:
#             # compute batch index for each row in tail_labels
#             # create tensor of batch ids repeated for each position in tail
#             batch_ids = torch.arange(B, device=device).unsqueeze(1).expand(B, L-pos).reshape(-1)[keep]
#             # set valid_mask[batch_ids, pos, kept_indices] = True
#             valid_mask[batch_ids, pos, kept_indices] = True

#     # Now for each position we may have zero valid tokens (should be ignored)
#     has_valid = valid_mask.any(dim=-1)  # (B, L) bool

#     # For positions with no valid tokens, set masked logits to -inf so logsumexp -> -inf.
#     neg_inf = -1e9
#     masked_log_probs = torch.where(valid_mask, log_probs, neg_inf)  # (B, L, V)

#     # Compute logsumexp over vocab dim -> log probability mass on valid tokens
#     log_sum = torch.logsumexp(masked_log_probs, dim=-1)  # (B, L)

#     # Positions without valid tokens will have log_sum approx = neg_inf. We mask them out.
#     valid_log_sum = log_sum[has_valid]  # 1D tensor of only valid positions

#     if valid_log_sum.numel() == 0:
#         # no valid positions: return zero or raise depending on preference
#         return torch.tensor(0.0, device=device, requires_grad=True)

#     loss = - valid_log_sum.mean()


#     # if print_debug:
#     #     print("=== DEBUG (loop version) ===")
#     #     print("logits.shape:", logits.shape)
#     #     print("shift_labels:\n", shift_labels)
#     #     print("valid_mask.sum per pos:", valid_mask.sum(-1))
#     #     print("has_valid:", has_valid)
#     #     print("log_sum:", log_sum)
#     #     print("loss:", loss.item())
#     return loss

"""
VECTORIZED LOSS: Using Upper-Triangular Masks

Key insight: For each position i, valid targets are ALL tokens at positions j≥i

Example sequence: [BOS, angle, hold1, hold2, hold3, EOS]
Position:           0     1      2      3      4     5

Future mask (upper triangular, each row shows what position i can see):
     j:  0  1  2  3  4  5
   i=0 [[1, 1, 1, 1, 1, 1],  ← pos 0 sees all future
   i=1  [0, 1, 1, 1, 1, 1],  ← pos 1 sees 1-5
   i=2  [0, 0, 1, 1, 1, 1],  ← pos 2 sees 2-5
   i=3  [0, 0, 0, 1, 1, 1],  ← pos 3 sees 3-5
   i=4  [0, 0, 0, 0, 1, 1],  ← pos 4 sees 4-5
   i=5  [0, 0, 0, 0, 0, 1]]  ← pos 5 sees 5 only
"""

def any_of_next_token_loss_vectorized(logits, shift_labels, ignore_index=-100, print_debug=False):
    """
    logits: (B, L, V) # batch_size, seq_len, vocab_size
    shift_labels: (B, L)
    """
    B, L, V = logits.shape
    device = logits.device

    # Compute log probabilities
    log_probs = F.log_softmax(logits, dim=-1)  # (B, L, V)

    # mask for valid label positions
    # [TRUE, TRUE, TRUE, TRUE, FALSE, FALSE, ...] where FALSE are PADs
    valid_positions = (shift_labels != ignore_index)  # (B, L)


    # upper-triangle of 1 size (L, L)
    future_mask = torch.triu(torch.ones((L, L), device=device, dtype=torch.bool))

    # Combine: for each sequence, each position sees all future tokens that are valid
    # Expand: valid_positions.unsqueeze(1): (B, L) → (B, 1, L)
    # valid_future_mask[b, i, j] = True if label[j] is valid future of i and j >= i
    valid_future_mask = valid_positions.unsqueeze(1) & future_mask  # (B, L, L)
    # valid_future_mask[0][0] => [TRUE, TRUE, TRUE, ..., TRUE, FALSE, ..., FALSE] (True for all valid tokens + False for PADs)
    # valid_future_mask[0][1] => [FALSE, TRUE, TRUE, ..., TRUE, FALSE, ..., FALSE]
    # valid_future_mask[0][2] => [FALSE, FALSE, TRUE, ..., TRUE, FALSE, ..., FALSE]
    # valid_future_mask[0][-seq_len] => [FALSE, FALSE, FALSE, ..., FALSE, FALSE, ..., FALSE]


    labels_expanded = shift_labels.unsqueeze(1).expand(-1, L, -1)  # (B, L, L)
    # labels_expanded[0] => (B, L) 2d array, row count = batch_size B=16
    # each row labels_expanded[0][0] is [token0, token1, token2, ..., token_seq_len, -100, -100, ...] (len L=21)

    # Set ignored ones to -1
    labels_expanded = torch.where(valid_future_mask, labels_expanded, -torch.ones_like(labels_expanded))
    # each row labels_expanded[0][0] is now [token0, token1, token2, ..., token_seq_len, -1, -1, ...] (len L=21)
    # labels_expanded[0][0] => [token0, token1, token2, ..., token_seq_len, -1, -1, ...] (len L=21)
    # labels_expanded[0][1] => [-1,     token1, token2, ..., token_seq_len, -1, -1, ...]
    # labels_expanded[0][2] => [-1,       -1,   token2, ..., token_seq_len, -1, -1, ...]


    # Build boolean mask per vocab id using scatter_
    valid_token_mask = torch.zeros((B, L, V), dtype=torch.bool, device=device)
    scatter_idx = labels_expanded.clone()
    scatter_idx[scatter_idx < 0] = 0  # convert -1 to dummy 0, to ignore the positions

    valid_token_mask.scatter_(dim=2, index=scatter_idx, src=valid_future_mask)  # mark valid vocab positions
    # (B, L, V)(16, 21, 1932)
    # valid_token_mask[0][0]
    # [PAD, BOS, EOS, UNK, ---angle---, ---grade---, ---holds---]
    # valid_token_mask[0][0], sum=12 < seq_len
    # [FALSE, TRUE, True, ---one True angle, ---one True grade, ---several True holds---]

    # valid_token_mask[0][1], sum=11
    # [FALSE, FALSE, True, ---one True angle, ---one True grade, ---several True holds---]

    # valid_token_mask[0][2], sum=10
    # [FALSE, FALSE, FALSE, ---one True angle, ---one True grade, ---several True holds---]

    # valid_token_mask[0][3], sum=9
    # [FALSE, FALSE, FALSE, ---all False angle, ---one True grade, ---several True holds---]

    # valid_token_mask[0][4], sum=8
    # [FALSE, FALSE, FALSE, ---all False angle, ---all False grade, ---several True holds---]

    # and then the hold tokens ...

    # note, for next example in batch:
    # valid_token_mask[1][0], sum=17 < seq_len
    # [FALSE, TRUE, True, ---one True angle, ---one True grade, ---several True holds---]


    valid_token_mask[:, :, 0] &= (labels_expanded[:, :, 0] != 0)  # fix dummy zeros if any


    # Filter(/mask) log_probs for only valid positions
    masked_log_probs = torch.where(valid_token_mask, log_probs, torch.full_like(log_probs, -1e9))
    # log_probs[0][0] =>        [-7.6137, -6.4946, -7.7223, ...] the usual
    # valid_token_mask[0][0] => [FALSE,    TRUE,    TRUE,   ...]
    # masked_log_probs[0][0] => [-1e9,    -6.4946, -7.7223, ...]


    # Combine log(P(token1) + P(token2) + ... + P(tokenN))
    log_sum = torch.logsumexp(masked_log_probs, dim=-1)  # (B, L)


    # Only keep positions that had at least one valid target
    has_valid = valid_token_mask.any(dim=-1) # Shape: (B, L)

    # average negative log-likelihood
    loss = -log_sum[has_valid].mean()


    return loss


# gpt

In [ ]:
class KilterGPT(nn.Module):
    """GPT-2 model for generating Kilter Board climbing routes."""
    def __init__(
        self,
        vocab_size: int,
        n_embd: int = 192,
        n_head: int = 3,
        n_layer: int = 3,
        n_positions: int = 128,
        dropout: float = 0.1
        ):
        super().__init__()
        config = GPT2Config(
            vocab_size=vocab_size,
            n_embd=n_embd,
            n_head=n_head,
            n_layer=n_layer,
            n_positions=n_positions,
            n_ctx=n_positions,
            resid_pdrop=dropout,
            embd_pdrop=dropout,
            attn_pdrop=dropout,
        )
        self.model = GPT2LMHeadModel(config)
        self.config = config

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=None)

        logits = outputs.logits  # (B, L, V) (batch_size, seq_len, vocab_size)

        # shift predictions and labels for next-token prediction
        shift_logits = logits[..., :-1, :].contiguous()   # predict token at t+1 using tokens up to t
        shift_labels = labels[..., 1:].contiguous()       # label for position t is token at t+1

        loss = any_of_next_token_loss(shift_logits, shift_labels, ignore_index=-100)
        outputs.loss = loss

        return outputs

In [ ]:
model = KilterGPT(
    vocab_size=tokenizer.vocab_size,
    n_embd=256,
    n_head=4,
    n_layer=6,
    n_positions=128,
    dropout=0.1
    )
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=3,
    overwrite_output_dir=True,
    logging_steps=100,  # Log more frequently for wandb
    eval_steps=1-00,
    save_steps=1000,
    num_train_epochs=30,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    weight_decay=0.01,
    adam_beta1=0.9,
    adam_beta2=0.999,
    report_to="wandb",  # Enable wandb reporting
    remove_unused_columns=False,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_dir=f"{OUT_DIR}/logs",
    load_best_model_at_end=True,
    dataloader_pin_memory=False,
    run_name='run_name',  # Use the same run name
    )
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=datasets["train"],
    eval_dataset=datasets["val"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
    )

# debug loss

In [ ]:
device = next(model.parameters()).device
# INSPECTIGN THE FIRST BATCH (with size B=16)
batch = next(iter(trainer.get_train_dataloader()))
# print({k: v.shape for k, v in batch.items()})

batch = {k: v.to(device) for k, v in batch.items()}

model.eval()
with torch.no_grad():
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"],
    )

logits = outputs.logits.detach().cpu()
labels = batch["labels"].cpu()

# loss_loop = any_of_next_token_loss(logits, labels, print_debug=True)
loss_vec  = any_of_next_token_loss_vectorized(logits, labels, print_debug=True)
loss_vec

In [ ]:
vocab_size = 6
seq_len = 3
batch_size = 1

logits = torch.tensor([
    [
        [1.0, 2.0, 3.0, 0.0, -1.0, 0.5],   # position 0
        [2.0, 1.0, 0.5, -0.5, 3.0, 0.0],   # position 1
        [0.0, 1.0, 0.0, 2.0, -1.0, 3.0]    # position 2
    ]
], dtype=torch.float)

# labels for the same sequence (token ids)
# Shape: (1, 3)
labels = torch.tensor([[1, 3, 5]])

# 🔥 Run the loss with debug prints
loss = any_of_next_token_loss_vectorized(logits, labels, print_debug=False)
loss

In [ ]:

# Get the train dataloader from your Trainer
train_dataloader = trainer.get_train_dataloader()

# Grab one batch
batch = next(iter(train_dataloader))

# Inspect what’s inside
for k, v in batch.items():
    print(k, v.shape)

# Move model to device and set to eval
model = trainer.model
device = next(model.parameters()).device
model.eval()

# Move batch tensors to device
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

# Forward pass (disable gradient)
with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )

# Extract logits
logits = outputs.logits  # shape (B, L, V)
print(f"logits shape: {logits.shape}")

# Also inspect a slice
print("Example logits[0, 0, :10]:", logits[0, 0, :10])
print("Example labels[0, :10]:", labels[0, :10])

# Optionally, compute your custom loss
shift_logits = logits[..., :-1, :].contiguous()
shift_labels = labels[..., 1:].contiguous()

loss = any_of_next_token_loss_vectorized(shift_logits, shift_labels, print_debug=False)
print("Custom loss:", loss.item())


In [ ]:
import torch.nn.functional as F

idx = 3
input_decoded = tokenizer.decode(input_ids[idx], skip_special_tokens=True)
print(f"\n🧩 Input sequence:\n{input_decoded}\n")

# Compute probabilities for this example
probs = F.softmax(logits[idx], dim=-1)
topk = torch.topk(probs, 3, dim=-1)  # top-3 predictions at each position

print("🔍 Token-level predictions (first 10 tokens):\n")
for i in range(10):  # first 10 tokens
    target_id = labels[idx, i].item()
    target_str = tokenizer.decode([target_id])

    preds_ids = topk.indices[i]
    preds_probs = topk.values[i]
    preds = [(tokenizer.decode([pid.item()]), float(pprob)) for pid, pprob in zip(preds_ids, preds_probs)]

    preds_display = ", ".join([f"{tok} ({p:.5f})" for tok, p in preds])
    print(f"Target: {target_str:10} | Top-3: {preds_display}")


# augment
- in dataset, usually, start holds come first
- e.g. sequence that is actually climbed: [BOS, angle40, grade15, start111, start999, hand222, foot444, hand333, finish777, EOS, (PAD)]
- but the model should be able to generate like this:
- given sequence [BOS, angle40, grade15, hand222, foot444]
    -  predict any of [start111, start999, and333, finish777, EOS] since they all complete the climbing route
    - order in the token is invariant(irrelavant) to the climbs after decoding
- therefore, training set should have shuffled sequence of holds as new augmented examples
- we'll skip [BOS, grade, angle, EOS] and shuffle the holds, append to the train set, maybe 2-3x

In [ ]:
import random
from copy import deepcopy

def augment_shuffle_routes(dataset, num_augments=2, tokenizer=None):

    """Return augmented dataset with shuffled holds."""
    augmented = []

    for example in dataset:
        tokens = example["input_ids"]

        # Find EOS (if any)
        eos_idx = tokens.index(tokenizer.eos_token_id) if tokenizer.eos_token_id in tokens else len(tokens)

        prefix = tokens[:3]  # [BOS, angle, grade]
        holds = tokens[3:eos_idx]
        tail = tokens[eos_idx:]  # [EOS, PAD, PAD, ...]

        for _ in range(num_augments): # 2 more augmentations per example
            shuffled = random.sample(holds, len(holds))  # new random order
            new_tokens = prefix + shuffled + tail
            new_example = deepcopy(example)
            new_example["input_ids"] = new_tokens
            if "labels" in example:
                new_example["labels"] = new_tokens.copy()
            augmented.append(new_example)

    full_dataset = dataset.to_list() + augmented
    return Dataset.from_list(full_dataset)

train_dataset = datasets["train"]
train_aug = augment_shuffle_routes(train_dataset, num_augments=3, tokenizer=tokenizer)
trainer.train_dataset = train_aug
# trainer.train() #works already but will augment on-the-fly
train_aug

# on-the-fly augment

In [ ]:
class AugmentedRouteDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset, tokenizer, num_augments=3):
        self.ds = hf_dataset
        self.tokenizer = tokenizer
        self.num_augments = num_augments
        self.n = len(hf_dataset)

    def __len__(self):
        return self.n * (self.num_augments + 1)  # originals + N augments

    def __getitem__(self, idx):
        base_idx = idx % self.n      # which real example in the base dataset
        aug_idx  = idx // self.n     # which augmentation slot (0 = original)

        example = self.ds[base_idx]
        tokens = example["input_ids"]

        if aug_idx > 0:  # 0 = original, others = augmented
            eos_idx = tokens.index(self.tokenizer.eos_token_id) if self.tokenizer.eos_token_id in tokens else len(tokens)
            prefix = tokens[:3]
            holds = tokens[3:eos_idx]
            tail = tokens[eos_idx:]

            random.shuffle(holds)          # in-place
            tokens = prefix + holds + tail  # use the shuffled list


        max_len = tokenizer.model_max_length  # or a fixed number

        example["input_ids"] = tokens + [tokenizer.pad_token_id] * (max_len - len(tokens))
        example["labels"] = tokens + [-100] * (max_len - len(tokens)) #ignore_index for padding
        example["token_type_ids"] = [0] * max_len
        example["attention_mask"] = [1] * len(tokens) + [0] * (max_len - len(tokens))


        return example

augmented_dataset = AugmentedRouteDataset(datasets["train"], tokenizer, num_augments=3)
trainer.train_dataset = augmented_dataset

In [ ]:
trainer.train()